# Objective O3 – Optimization Demo

**Sleep-Based Low-Latency Access for M2M Communications**

This notebook demonstrates Objective O3: optimising sleep and access parameters for latency–longevity trade-offs.

## Contents
1. [Setup](#1-Setup)
2. [Task 3.1a – Grid Search: optimal q for max lifetime](#2-Task-31a)
3. [Task 3.1b – Grid Search: optimal q for min delay](#3-Task-31b)
4. [Task 3.1c – Pareto Tradeoff Curve (varying ts)](#4-Task-31c)
5. [Task 3.1d – 2-D Heatmap (q × ts)](#5-Task-31d)
6. [Task 3.2a – Prioritization Scenario Comparison](#6-Task-32a)
7. [Task 3.2b – On-Demand Sleep vs Duty Cycling](#7-Task-32b)
8. [Design Guidelines Summary](#8-Design-Guidelines)

## 1 Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams.update({'figure.dpi': 110, 'font.size': 11})

from src.simulator import SimulationConfig
from src.power_model import PowerModel, PowerProfile
from src.optimization import (
    ParameterOptimizer,
    DutyCycleSimulator,
    PrioritizationAnalyzer,
    OptimizationVisualizer,
)

print('Imports OK')

In [ ]:
# ── Base configuration ────────────────────────────────────────────────────
# n=20 nodes, λ=0.01, ts=10, tw=5 – matches the O2 baseline.
# Adjust QUICK_MODE=True to run faster (fewer replications / values).

QUICK_MODE = True   # Set False for publication-quality results

N_REPS   = 5  if QUICK_MODE else 15
MAX_SLOTS = 30_000 if QUICK_MODE else 60_000

base_config = SimulationConfig(
    n_nodes=20,
    arrival_rate=0.01,
    transmission_prob=0.05,
    idle_timer=10,
    wakeup_time=5,
    initial_energy=5_000,
    power_rates=PowerModel.get_profile(PowerProfile.GENERIC_LOW),
    max_slots=MAX_SLOTS,
    seed=None,
)

print(f'n={base_config.n_nodes}, λ={base_config.arrival_rate}, '
      f'ts={base_config.idle_timer}, tw={base_config.wakeup_time}')
print(f'Quick mode: {QUICK_MODE}  |  reps/config: {N_REPS}  |  max_slots: {MAX_SLOTS:,}')

## 2 Task 3.1a – Grid Search q for Max Lifetime

Sweep transmission probability q and find q* that maximises mean node lifetime.
Higher q increases throughput but also drains energy faster through collisions.

In [ ]:
q_values = list(np.linspace(0.01, 0.40, 12 if QUICK_MODE else 20))

opt_lt = ParameterOptimizer.grid_search_q(
    base_config,
    q_values=q_values,
    objective='lifetime',
    n_replications=N_REPS,
    verbose=True,
)

print(f'\n→ Optimal q* for MAX LIFETIME: {opt_lt.optimal_value:.4f}')
print(f'  Max mean lifetime: {opt_lt.optimal_metric:.4f} years')

## 3 Task 3.1b – Grid Search q for Min Delay

In [ ]:
opt_d = ParameterOptimizer.grid_search_q(
    base_config,
    q_values=q_values,
    objective='delay',
    n_replications=N_REPS,
    verbose=True,
)

print(f'\n→ Optimal q* for MIN DELAY: {opt_d.optimal_value:.4f}')
print(f'  Min mean delay: {opt_d.optimal_metric:.2f} slots  ({opt_d.optimal_metric*6:.1f} ms)')

In [ ]:
# ── Plot lifetime and delay vs q, marking both optima ─────────────────────
fig, ax_l, ax_r = OptimizationVisualizer.plot_q_sweep(
    q_values=opt_lt.all_values,
    lifetimes=opt_lt.all_lifetimes,
    delays=opt_lt.all_delays,
    optimal_q_lifetime=opt_lt.optimal_value,
    optimal_q_delay=opt_d.optimal_value,
    title='Lifetime and Delay vs Transmission Probability q  (ts=10)',
)
plt.savefig('opt_q_sweep.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: opt_q_sweep.png')

**Key observation:** Lifetime peaks at an intermediate q* (≈ 1/n for the baseline n=20),
while delay decreases monotonically with q up to the point where collisions dominate.
These two optima differ, quantifying the fundamental latency–longevity tension.

## 4 Task 3.1c – Pareto Tradeoff Curve (varying ts)

For each idle-timer value ts, sweep q and record the point that maximises lifetime.
Plotting these points traces the Pareto frontier: longer ts → more lifetime, more delay.

In [ ]:
ts_values = [1, 10, 50] if QUICK_MODE else [1, 5, 10, 20, 50]
q_sweep   = list(np.linspace(0.01, 0.40, 10 if QUICK_MODE else 15))

tradeoff_pts = ParameterOptimizer.tradeoff_analysis(
    base_config,
    q_values=q_sweep,
    ts_values=ts_values,
    n_replications=N_REPS,
    verbose=True,
)

print('\nPareto points (ts, q*, lifetime, delay):')
for pt in tradeoff_pts:
    print(f'  ts={pt.ts:>3},  q*={pt.q:.3f},  L={pt.lifetime_years:.4f}y,  T={pt.delay_slots:.1f} slots ({pt.delay_ms:.0f} ms)')

In [ ]:
fig, ax = OptimizationVisualizer.plot_pareto_frontier(
    tradeoff_pts,
    title='Pareto Frontier: Max Achievable Lifetime vs Min Delay for varying ts',
)
plt.savefig('opt_pareto_frontier.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: opt_pareto_frontier.png')

**Interpretation:** Each point represents the best (max-lifetime) operating point for a given ts.
Moving from low ts (left, low delay) to high ts (right, high delay) yields longer lifetimes
– the classic latency–longevity trade-off from the paper's Sec IV.

## 5 Task 3.1d – 2-D Grid Search (q × ts)

In [ ]:
q_2d  = [0.01, 0.05, 0.10, 0.20, 0.30] if QUICK_MODE else [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
ts_2d = [1, 10, 50]                     if QUICK_MODE else [1, 5, 10, 20, 50]

grid = ParameterOptimizer.grid_search_q_ts(
    base_config,
    q_values=q_2d,
    ts_values=ts_2d,
    n_replications=max(N_REPS - 2, 3),
    verbose=True,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

OptimizationVisualizer.plot_tradeoff_heatmap(grid, metric='lifetime',
    title='Mean Lifetime (years) over (q, ts) Grid', ax=axes[0])

OptimizationVisualizer.plot_tradeoff_heatmap(grid, metric='delay',
    title='Mean Delay (slots) over (q, ts) Grid', ax=axes[1])

fig.suptitle('2-D Parameter Space: (q, ts)', fontsize=14)
plt.tight_layout()
plt.savefig('opt_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: opt_heatmaps.png')

In [ ]:
# Print the (q, ts) combination with max lifetime and min delay
lt_mat  = grid['lifetime_matrix']
d_mat   = grid['delay_matrix']
q_vals  = grid['q_values']
ts_vals = grid['ts_values']

best_lt_idx = np.unravel_index(np.argmax(lt_mat), lt_mat.shape)
best_d_idx  = np.unravel_index(np.argmin(d_mat[d_mat > 0].min() == d_mat, axis=None if d_mat.ndim == 0 else None), d_mat.shape)

# Recompute properly
positive_d = np.where(d_mat > 0, d_mat, np.inf)
best_d_idx = np.unravel_index(np.argmin(positive_d), positive_d.shape)

print('Global optimum – Max Lifetime:')
print(f'  ts={ts_vals[best_lt_idx[0]]}, q={q_vals[best_lt_idx[1]]:.3f}  →  {lt_mat[best_lt_idx]:.4f} years')
print('Global optimum – Min Delay:')
print(f'  ts={ts_vals[best_d_idx[0]]}, q={q_vals[best_d_idx[1]]:.3f}  →  {d_mat[best_d_idx]:.1f} slots ({d_mat[best_d_idx]*6:.0f} ms)')

## 6 Task 3.2a – Prioritization Scenario Comparison

Compare three canonical configurations:

| Scenario | ts | q | Description |
|---|---|---|---|
| Low-Latency Priority | 1 | 2/n | Small ts → minimal sleep delay |
| Balanced | 10 | 1/n | Moderate – Aloha-optimal q |
| Battery-Life Priority | 50 | 0.5/n | Large ts → max time in sleep |

In [ ]:
comparison = PrioritizationAnalyzer.run_scenario_comparison(
    n_nodes=20,
    arrival_rate=0.01,
    max_slots=MAX_SLOTS,
    n_replications=N_REPS * 2,
    verbose=True,
)

PrioritizationAnalyzer.print_comparison_summary(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

OptimizationVisualizer.plot_prioritization_comparison(
    comparison,
    title='Scenario Comparison: Delay vs Lifetime',
    ax=axes[0],
)

OptimizationVisualizer.plot_gains_losses_bar(
    comparison,
    title='% Gains/Losses vs Balanced Baseline',
    ax=axes[1],
)

plt.tight_layout()
plt.savefig('opt_prioritization_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: opt_prioritization_comparison.png')

In [ ]:
# Summarise numerical gains
print('Quantified Gains/Losses vs Balanced Baseline:\n')
for name, g in comparison.gains_vs_baseline.items():
    dc  = g['delay_change_pct']
    lc  = g['lifetime_change_pct']
    d_s = g['delay_slots']
    lt  = g['lifetime_years']
    lt_str = f'{lt:.4f}' if lt != float('inf') else 'inf'
    print(f'  {name}')
    print(f'    Delay:    {d_s:.1f} slots ({d_s*6:.0f} ms)  → {dc:+.1f}% vs balanced')
    print(f'    Lifetime: {lt_str} yr  → {lc:+.1f}% vs balanced')

## 7 Task 3.2b – On-Demand Sleep vs Duty Cycling

On-demand sleep enters sleep only when the queue is empty (after ts idle slots).
Duty cycling uses a fixed periodic awake/sleep schedule regardless of traffic.

The paper predicts on-demand sleep outperforms duty cycling for M2M traffic
because it avoids unnecessary wakeups when there is no traffic.

In [ ]:
ts_dc    = [1, 10, 50] if QUICK_MODE else [1, 5, 10, 20, 50]
fracs_dc = [0.1, 0.3, 0.7] if QUICK_MODE else [0.1, 0.2, 0.3, 0.5, 0.7]

dc_comparison = DutyCycleSimulator.compare_with_on_demand(
    base_config,
    ts_values=ts_dc,
    awake_fractions=fracs_dc,
    cycle_period=20,
    n_replications=N_REPS,
    verbose=True,
)

In [ ]:
fig, ax = OptimizationVisualizer.plot_duty_cycle_comparison(
    dc_comparison,
    title='On-Demand Sleep vs Duty Cycling: Lifetime–Delay Tradeoff',
)
plt.savefig('opt_duty_cycle_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: opt_duty_cycle_comparison.png')

## 8 Design Guidelines Summary

Based on the O3 experiments above:

In [ ]:
# Determine optimal q for max lifetime from grid search
opt_q_star = opt_lt.optimal_value
analytical_q_star = 1.0 / base_config.n_nodes

print('=' * 65)
print('DESIGN GUIDELINES  (n=20, λ=0.01, 6ms slots, E=5000 units)')
print('=' * 65)

print(f"""
1. OPTIMAL TRANSMISSION PROBABILITY
   • Analytical optimum (Aloha):  q* = 1/n = {analytical_q_star:.4f}
   • Simulated optimum (max lt):  q* ≈ {opt_q_star:.4f}
   • Rule: set q ≈ 1/n for the best lifetime-throughput balance.

2. IDLE TIMER SELECTION
   • Small ts (ts=1):  minimal wake-up delay, lowest lifetime.
   • Large ts (ts=50): maximum lifetime, higher queuing delay.
   • Rule: for λ=0.01, ts=10 gives a good balanced operating point.
     For latency-critical apps (delay < 100 ms ≈ 17 slots), use ts ≤ 5.
     For 10-year lifetime targets, use ts ≥ 20 with low q.

3. PRIORITIZATION
   • Low-latency priority (ts=1, q=2/n): reduces delay significantly,
     at a meaningful lifetime cost – check gains_vs_baseline above.
   • Battery-life priority (ts=50, q=0.5/n): maximises lifetime but
     increases delay – use only when latency requirements are loose.

4. SLEEP STRATEGY
   • On-demand sleep outperforms duty cycling for sparse M2M traffic
     (λ << 1) because it avoids unnecessary wakeups.
   • Use duty cycling only when a guaranteed response time window is
     needed (e.g., fixed polling cycle for actuators).
""")

print('=' * 65)